In [8]:
%%bash
sudo apt-get update &> /dev/null
sudo apt-get install -y pciutils 1> /dev/null
curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...


>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [13]:
%%bash
pip install -q "polars[gpu]" \
    spacy \
    seaborn \
    textstat \
    ollama \
    matplotlib \
    pyarrow \
    typing-extensions \
    pyarrow==24.0.0 \
    seaborn==0.13.2 \
    spacy==3.8.13 \
    textstat==0.7.13 \
    transformers==5.10.2 \
    click==8.0.4 \
    click-completion>=0.5.2

spacy download en_core_web_sm

  Using cached en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [15]:
import os
import re
import json
import numpy
import polars
import ollama
import spacy
import seaborn
import textstat
from pathlib import Path
from itertools import permutations
from matplotlib import pyplot
from capstone_lib import CapstoneLib 

BASE_PATH = "/workspace"
os.chdir(f"{BASE_PATH}/uol_capstone_project")


In [20]:
# 1  Cuad Data loading
PATH_CUAD_PARQUET = Path(BASE_PATH).rglob("**/*/0001_cuad_dataset.parquet")
PATH_CUAD_JSON = Path(BASE_PATH).rglob("**/*/CUADv1.json")

CUAD_PARQUET = next(PATH_CUAD_PARQUET, None)
CUAD_JSON = next(PATH_CUAD_JSON, None)


if os.path.exists(CUAD_PARQUET):
    cuad_legal = CapstoneLib.load_parquet(CUAD_PARQUET)

    with open(CUAD_JSON) as cuad:
        cuad_json = json.load(cuad.buffer)

    legal_pieces = {}

    for data in cuad_json.get("data"):
        full_context = ""
        for paragraph in data.get("paragraphs"):
            if context := paragraph.get("context", ""):
                full_context += context
        legal_pieces[data.get("title")] = full_context

    cuad_legal = polars.DataFrame(legal_pieces)
    cuad_legal = cuad_legal.transpose(include_header=True)
    cuad_legal.columns = ["title", "paragraphs"]

    cuad_legal.write_parquet("parquet_files/0001_cuad_dataset.parquet")


In [ ]:
# 2 Cuad data parsing

# Here we are trying to load cached file if it exists, otherwise we will need to reprocess everything
PATH_EXTRACTED_SENTENCES = Path(".").rglob("**/*/0002_cuad_dataset.parquet")
CUAD_EXTRACTED = next(PATH_EXTRACTED_SENTENCES, "non_existent.parquet")

nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")

if os.path.exists(CUAD_EXTRACTED):
    cuad_legal = polars.read_parquet(CUAD_EXTRACTED)
else:
    cuad_legal = cuad_legal.with_columns(
        polars.col("paragraphs").map_elements(
            CapstoneLib.paragraphs_extractor, return_dtype=polars.List(polars.String)
        ).alias("paragraphs")
    )

    all_paragraphs = cuad_legal["paragraphs"].explode().to_list()
    docs = nlp.pipe(all_paragraphs, batch_size=64, n_process=4)
    clauses = [CapstoneLib.extract_clauses_from_doc(doc) for doc in docs]
    cuad_legal = cuad_legal.with_columns(
        polars.Series("sentences", clauses)
    )

    cuad_legal.write_parquet(f"parquet_files/0002_cuad_dataset.parquet")

In [ ]:
# 3 apply the nine metrics

cuad_legal = cuad_legal.explode("sentences", empty_as_null=False)

cuad_legal = CapstoneLib.add_complexity_features(
    cuad_legal,
    nlp,
)

In [ ]:
# Parse unique clauses only, then join back on the clause text
PATH_EXTRACTED_SENTENCES = Path(".").rglob("**/*/0002_cuad_dataset.parquet")
CUAD_EXTRACTED = next(PATH_EXTRACTED_SENTENCES)

if os.path.exists(CUAD_EXTRACTED):
    cuad_legal = polars.read_parquet(CUAD_EXTRACTED)

else:
    unique_clauses = (
        cuad_legal["sentences"]
        .unique()
        .to_list()
    )
    
    docs = list(
        nlp.pipe(
            unique_clauses,
            batch_size=512,
            n_process=3,
        )
    )
    
    features = [
        CapstoneLib.syntax_features(doc)
        for doc in docs
    ]
    
    syntax_df = polars.DataFrame({
        "sentences": unique_clauses,
        "mdd": [f[0] for f in features],
        "tree_depth": [f[1] for f in features],
        "clause_density": [f[2] for f in features],
    })
    
    cuad_legal = cuad_legal.join(
        syntax_df,
        on="sentences",
        how="left"
    )
    
    cuad_legal.select(
        ["clause_density", "mdd", "tree_depth"]
    ).describe()
    
    cuad_legal.write_parquet(f"./parquet_files/{FILENAME}")